# SmartDs 2D Slice Plot (Starter)

This notebook reads an already-2D BATSRUS slice file and plots a **flat-shaded triangulated field** on the native 2D mesh.

It reuses the library (`SmartDs`) and the file's native mesh connectivity (`sds.corners`) and does **not** resample.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.data.samples import get_sample
from starwinds_analysis.visualisation.slice import auto_coords, triangles

plt.rcParams['figure.dpi'] = 120


## Choose a 2D File

By default this looks for a `z=0*.plt` slice in `sample_data/`.

In [ ]:
DATA_FILE = get_sample('z=0_var_3_n00060000.plt', echo=True)
# DONE this should not be neede; the get_sampe should print for us.

sds = SmartDs.from_file(DATA_FILE)
print(sds)


In [ ]:
# Inspect available fields
", ".join(sds.variables)

## Auto-Detect Slice Coordinates and Choose a Field

The notebook inspects `X [R]`, `Y [R]`, `Z [R]` and picks the two coordinates that vary across the 2D slice.

In [ ]:
X_FIELD, Y_FIELD = auto_coords(sds)
C_FIELD = 'Rho [g/cm^3]'
tris = triangles(sds, X_FIELD, Y_FIELD)
print(f"Coordinates: {X_FIELD}, {Y_FIELD}")

fig, ax = plt.subplots(figsize=(7, 6))
img = ax.tripcolor(tris, sds.variable(C_FIELD), shading='flat', cmap='viridis', norm="log")
ax.triplot(tris, color='k', linewidth=0.15, alpha=0.15)
ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title(C_FIELD)
fig.colorbar(img, ax=ax, label=C_FIELD)
plt.show()


## Alfvén Surface (2D Contour)

This uses the BATSRUS recipe graph to compute the Alfvén Mach number `M_A` on demand and draws the `M_A = 1` contour as a 2D Alfvén-surface proxy in the slice plane.

In [ ]:
sds.add_batsrus_graph()
ma = sds.variable('M_A [none]')

fig, ax = plt.subplots(figsize=(7, 6))
img = ax.tripcolor(
    tris,
    ma,
    shading='flat',
    cmap='cividis',
    norm='log',
    vmin=1e-2,
    vmax=1e2,
)
img.cmap.set_under(img.cmap(0.0))
img.cmap.set_over(img.cmap(1.0))

ax.tricontour(tris, ma, levels=[1.0], colors='crimson', linewidths=1.5)
ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title('Alfvén Mach number')
fig.colorbar(img, ax=ax, label='M_A', extend='both')
plt.show()